In [1]:
import tensorflow as tf
import os
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

"""
CATTLE TRAINING 
"""

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)
# GPU setup
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except:
        pass

2026-02-05 12:34:53.370987: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 12:34:53.507249: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-05 12:34:53.550567: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-05 12:34:54.580060: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; 

In [2]:
dataset_root = "/home/student/Desktop/Kumar/dataset"

In [3]:
class CFG:
    IMG_SIZE = 224
    EMBED_DIM = 512
    MARGIN = 0.5
    SCALE = 64
    BATCH_SIZE = 32
    CENTER_LOSS_WEIGHT = 0.0001
    UNKNOWN_LOSS_WEIGHT = 0.1
    STEPS_PER_EPOCH = 200
    AUTOTUNE = tf.data.AUTOTUNE
    TRAIN_SPLIT = 0.85  # 85% train, 15% val
    
    # Curriculum schedule
    PHASE_1_EPOCHS = 5   # Warmup: 100% known
    PHASE_2_EPOCHS = 15  # Introduction: 80% known, 20% unknown
    PHASE_3_EPOCHS = 10  # Hardening: 60% known, 40% unknown
    
    TOTAL_EPOCHS = PHASE_1_EPOCHS + PHASE_2_EPOCHS + PHASE_3_EPOCHS

In [4]:
# DATA LOADING

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img
def augment_image(img):
    """Standard augmentation for known samples"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return img
def augment_unknown(img):
    """Heavy augmentation to create synthetic unknowns"""
    # Heavy color shifts
    img = tf.image.random_brightness(img, 0.5)
    img = tf.image.random_contrast(img, 0.5, 1.5)
    img = tf.image.random_saturation(img, 0.5, 1.5)
    img = tf.image.random_hue(img, 0.2)
    
    # Random flips
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    
    # Proper rotation
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    
    # Add noise
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=0.1)
    img = img + noise
    img = tf.clip_by_value(img, -2.0, 2.0)
    
    return img

class CombinedDatasetBuilder:
    """Combines all cows from face_train, face_test, face_val"""
    def __init__(self, face_folders, split='train', train_split=0.85, seed=42):
        self.cow_data = {}
        self.split = split
        
        np.random.seed(seed)
        
        # Combine all folders
        for face_folder in face_folders:
            if not os.path.exists(face_folder):
                print(f"Warning: {face_folder} not found, skipping...")
                continue
            
            for cls in sorted(os.listdir(face_folder)):
                cls_path = os.path.join(face_folder, cls)
                if not os.path.isdir(cls_path):
                    continue
                
                imgs = [os.path.join(cls_path, f) for f in os.listdir(cls_path)
                       if f.lower().endswith((".jpg", ".jpeg", ".png"))]
                
                if len(imgs) > 0:
                    if cls in self.cow_data:
                        self.cow_data[cls].extend(imgs)
                    else:
                        self.cow_data[cls] = imgs
        
        # Now split each cow's images into train/val
        self.split_data = {}
        for cow_id, img_list in self.cow_data.items():
            img_list_shuffled = sorted(img_list)  # Sort for reproducibility
            indices = np.arange(len(img_list_shuffled))
            np.random.shuffle(indices)
            img_list_shuffled = [img_list_shuffled[i] for i in indices]
            
            n_train = int(len(img_list_shuffled) * train_split)
            
            if split == 'train':
                self.split_data[cow_id] = img_list_shuffled[:n_train]
            else:  # 'val'
                self.split_data[cow_id] = img_list_shuffled[n_train:]
        
        self.classes = sorted(list(self.split_data.keys()))
        self.cls_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.split_data.values())
    
    def get_tfdata_unbatched(self, shuffle=True, augment=False):
        face_paths = []
        labels = []
        
        for cow_id, face_list in self.split_data.items():
            for face_path in face_list:
                face_paths.append(face_path)
                labels.append(self.cls_to_idx[cow_id])
        
        ds = tf.data.Dataset.from_tensor_slices({
            'face_path': face_paths,
            'label': labels
        })
        
        def map_fn(sample):
            face_img = load_image(sample['face_path'])
            
            if augment:
                face_img = augment_image(face_img)
            
            return {
                'face_input': face_img,
                'label_input': sample['label']
            }, sample['label']
        
        ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        
        if shuffle:
            ds = ds.shuffle(2000, seed=42, reshuffle_each_iteration=True)
        
        ds = ds.repeat()
        return ds
# Load data
print(" Loading ALL cattle data from face_train, face_test, face_val...")
all_face_folders = [
    os.path.join(dataset_root, "face_train"),
    os.path.join(dataset_root, "face_test"),
    os.path.join(dataset_root, "face_val")
]

train_data = CombinedDatasetBuilder(all_face_folders, split='train', train_split=CFG.TRAIN_SPLIT)
val_data = CombinedDatasetBuilder(all_face_folders, split='val', train_split=CFG.TRAIN_SPLIT)
print(f"\n Loaded {train_data.num_classes} cows")
print(f"  Training:   {train_data.total_images} images ({CFG.TRAIN_SPLIT*100:.0f}%)")
print(f"  Validation: {val_data.total_images} images ({(1-CFG.TRAIN_SPLIT)*100:.0f}%)")
print(f"  All cows appear in both train and val")

 Loading ALL cattle data from face_train, face_test, face_val...

 Loaded 468 cows
  Training:   3774 images (85%)
  Validation: 952 images (15%)
  All cows appear in both train and val


In [5]:
# MODEL 

def build_face_only_model(num_classes, embedding_size=512, margin=0.5, scale=64):
    
    face_input = keras.Input(shape=(224, 224, 3), name="face_input")
    label_input = keras.Input(shape=(), dtype=tf.int32, name="label_input")
    
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        input_shape=(224, 224, 3),
        weights="imagenet"
    )
    
    # Proper BatchNorm handling
    for layer in backbone.layers[:-30]:
        layer.trainable = False
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    
    # training=True to allow BN adaptation
    face_feat = backbone(face_input, training=True)
    face_feat = layers.GlobalAveragePooling2D()(face_feat)
    
    x = layers.Dropout(0.4)(face_feat)
    
    x = layers.Dense(embedding_size, kernel_regularizer=keras.regularizers.l2(1e-4), 
                     name="embedding_dense")(x)
    embeddings = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1), 
                               name="l2_normalize")(x)
    
    class SafeArcFaceLayer(layers.Layer):
        def __init__(self, n_classes, margin=0.5, scale=64, **kwargs):
            super().__init__(**kwargs)
            self.n_classes = n_classes
            self.margin = margin
            self.scale = scale
            self.cos_m = tf.cos(margin)
            self.sin_m = tf.sin(margin)
        
        def build(self, input_shape):
            self.W = self.add_weight(
                name="arcface_weights",
                shape=(input_shape[0][-1], self.n_classes),
                initializer=keras.initializers.HeNormal(),
                trainable=True,
                regularizer=keras.regularizers.l2(1e-4)
            )
            super().build(input_shape)
        
        def call(self, inputs):
            embeddings, labels = inputs
            
            W_norm = tf.nn.l2_normalize(self.W, axis=0)
            embeddings_norm = tf.nn.l2_normalize(embeddings, axis=1)
            
            cos_theta = tf.matmul(embeddings_norm, W_norm)
            cos_theta = tf.clip_by_value(cos_theta, -1.0 + 1e-7, 1.0 - 1e-7)
            
            sin_theta = tf.sqrt(tf.maximum(1.0 - tf.square(cos_theta), 1e-7))
            target_logits = cos_theta * self.cos_m - sin_theta * self.sin_m
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            
            labels_safe = tf.where(known_mask, labels, tf.zeros_like(labels))
            onehot_raw = tf.one_hot(labels_safe, depth=self.n_classes, dtype=tf.float32)
            onehot = onehot_raw * tf.expand_dims(tf.cast(known_mask, tf.float32), axis=1)
            
            output = cos_theta * (1.0 - onehot) + target_logits * onehot
            output = output * self.scale
            
            return output
        
        def get_config(self):
            return {"n_classes": self.n_classes, "margin": self.margin, "scale": self.scale}
    
    logits = SafeArcFaceLayer(num_classes, margin=margin, scale=scale, 
                              name="arcface")([embeddings, label_input])
    
    model = keras.Model(
        inputs=[face_input, label_input],
        outputs=[logits, embeddings],
        name="cattle_face_model"
    )
    
    return model


print("BUILDING MODEL")

model = build_face_only_model(
    train_data.num_classes,
    embedding_size=CFG.EMBED_DIM,
    margin=CFG.MARGIN,
    scale=CFG.SCALE
)

print(f"\nArchitecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace")
print(f"  Margin: {CFG.MARGIN}, Scale: {CFG.SCALE}")
print(f"  Classes: {train_data.num_classes} cows")
print(f"  BatchNorm: Last 30 layers adaptive, early layers frozen")

BUILDING MODEL


2026-02-05 12:37:12.232157: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 12:37:13.004429: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4731 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:d8:00.0, compute capability: 7.5



Architecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace
  Margin: 0.5, Scale: 64
  Classes: 468 cows
  BatchNorm: Last 30 layers adaptive, early layers frozen


In [6]:
# Test normalization

print("\nTesting embedding normalization...")
test_batch = {
    'face_input': tf.random.normal((4, 224, 224, 3)),
    'label_input': tf.constant([0, 1, 2, 3], dtype=tf.int32)
}
_, test_emb = model(test_batch, training=False)
test_norms = tf.norm(test_emb, axis=1).numpy()
print(f"  Embedding norms: {test_norms}")
assert np.all(np.abs(test_norms - 1.0) < 0.01), "Embeddings not normalized!"
print(" Embeddings properly normalized")


Testing embedding normalization...


2026-02-05 12:37:27.605733: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8201


  Embedding norms: [0.99999994 0.99999994 1.         0.99999994]
 Embeddings properly normalized


In [7]:
# CENTER LOSS 

class CenterLossLayer(layers.Layer):
    def __init__(self, num_classes, embedding_dim, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim
        self.alpha = tf.constant(alpha, dtype=tf.float32)
    
    def build(self, input_shape):
        self.centers = self.add_weight(
            name="class_centers",
            shape=(self.num_classes, self.embedding_dim),
            initializer="zeros",
            trainable=False
        )
        super().build(input_shape)
    
    def _compute_and_update(self, embeddings, labels):
        labels = tf.cast(labels, tf.int32)
        known_mask = labels >= 0
        known_embeddings = tf.boolean_mask(embeddings, known_mask)
        known_labels = tf.boolean_mask(labels, known_mask)
        
        if tf.shape(known_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        centers_batch = tf.gather(self.centers, known_labels)
        diff = known_embeddings - centers_batch
        loss = tf.reduce_mean(tf.reduce_sum(tf.square(diff), axis=1))
        
        unique_labels, idx = tf.unique(known_labels)
        num_unique = tf.shape(unique_labels)[0]
        summed = tf.math.unsorted_segment_sum(known_embeddings, idx, num_unique)
        counts = tf.math.unsorted_segment_sum(tf.ones_like(idx, dtype=tf.float32), idx, num_unique)
        counts = tf.reshape(counts, (-1, 1))
        new_centers = summed / (counts + 1e-8)
        
        current_centers = tf.gather(self.centers, unique_labels)
        updated = self.alpha * current_centers + (1.0 - self.alpha) * new_centers
        
        scatter_idx = tf.expand_dims(unique_labels, axis=1)
        self.centers.assign(tf.tensor_scatter_nd_update(self.centers, scatter_idx, updated))
        
        return loss
    
    def call(self, embeddings, labels, training=True):
        return self._compute_and_update(embeddings, labels)

center_loss_layer = CenterLossLayer(train_data.num_classes, CFG.EMBED_DIM, alpha=0.5)
print(f"\nCenter Loss: weight={CFG.CENTER_LOSS_WEIGHT}, alpha=0.5")


Center Loss: weight=0.0001, alpha=0.5


In [8]:
# UNKNOWN REJECTION LOSS 

class UnknownRejectionLoss(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def call(self, embeddings, labels, class_weights):
        labels = tf.cast(labels, tf.int32)
        unknown_mask = tf.less(labels, 0)
        
        unknown_embeddings = tf.boolean_mask(embeddings, unknown_mask)
        
        if tf.shape(unknown_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        W_norm = tf.nn.l2_normalize(class_weights, axis=0)
        similarities = tf.matmul(unknown_embeddings, W_norm)
        max_similarities = tf.reduce_max(similarities, axis=1)
        loss = tf.reduce_mean(max_similarities)
        
        return loss

unknown_rejection_layer = UnknownRejectionLoss()
print(f"Unknown Rejection Loss: weight={CFG.UNKNOWN_LOSS_WEIGHT}")

Unknown Rejection Loss: weight=0.1


In [9]:
# TRAINER 

class FaceOnlyTrainer(keras.Model):
    def __init__(self, base_model, center_loss_layer, unknown_rejection_layer,
                 center_loss_weight, unknown_loss_weight, **kwargs):
        super().__init__(**kwargs)
        self.base_model = base_model
        self.center_loss_layer = center_loss_layer
        self.unknown_rejection_layer = unknown_rejection_layer
        self.center_loss_weight = center_loss_weight
        self.unknown_loss_weight = unknown_loss_weight
        
        self.arcface_loss_tracker = keras.metrics.Mean(name="arcface_loss")
        self.center_loss_tracker = keras.metrics.Mean(name="center_loss")
        self.unknown_loss_tracker = keras.metrics.Mean(name="unknown_loss")
        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.accuracy_tracker = keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    
    def call(self, inputs):
        return self.base_model(inputs)
    
    def train_step(self, data):
        inputs, labels = data
        
        with tf.GradientTape() as tape:
            logits, embeddings = self.base_model(inputs, training=True)
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            
            # ArcFace loss
            def compute_arcface():
                known_logits = tf.boolean_mask(logits, known_mask)
                known_labels = tf.boolean_mask(labels, known_mask)
                return tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=known_labels,
                        logits=known_logits
                    )
                )
            
            arcface_loss = tf.cond(
                tf.reduce_any(known_mask),
                compute_arcface,
                lambda: tf.constant(0.0, dtype=tf.float32)
            )
            
            # Center loss
            center_loss = self.center_loss_layer(embeddings, labels, training=True)
            
            # Unknown rejection loss
            arcface_layer = self.base_model.get_layer('arcface')
            class_weights = arcface_layer.W
            unknown_loss = self.unknown_rejection_layer(embeddings, labels, class_weights)
            
            # Total loss
            total_loss = (arcface_loss + 
                         self.center_loss_weight * center_loss +
                         self.unknown_loss_weight * unknown_loss)
            
            if self.base_model.losses:
                reg_loss = tf.add_n(self.base_model.losses)
            else:
                reg_loss = tf.constant(0.0, dtype=total_loss.dtype)
            
            total_loss = total_loss + reg_loss
        
        gradients = tape.gradient(total_loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.base_model.trainable_variables))
        
        self.arcface_loss_tracker.update_state(arcface_loss)
        self.center_loss_tracker.update_state(center_loss)
        self.unknown_loss_tracker.update_state(unknown_loss)
        self.total_loss_tracker.update_state(total_loss)
        
        def update_acc():
            known_logits = tf.boolean_mask(logits, known_mask)
            known_labels = tf.boolean_mask(labels, known_mask)
            self.accuracy_tracker.update_state(known_labels, known_logits)
            return tf.constant(0)
        
        tf.cond(tf.reduce_any(known_mask), update_acc, lambda: tf.constant(0))
        
        return {
            "loss": self.total_loss_tracker.result(),
            "arcface_loss": self.arcface_loss_tracker.result(),
            "center_loss": self.center_loss_tracker.result(),
            "unknown_loss": self.unknown_loss_tracker.result(),
            "accuracy": self.accuracy_tracker.result()
        }
    
    def test_step(self, data):
        inputs, labels = data
        
        logits, embeddings = self.base_model(inputs, training=False)
        labels = tf.cast(labels, tf.int32)
        
        arcface_loss = tf.reduce_mean(
            tf.nn.sparse_softmax_cross_entropy_with_logits(
                labels=labels,
                logits=logits
            )
        )
        
        center_loss = self.center_loss_layer(embeddings, labels, training=False)
        total_loss = arcface_loss + self.center_loss_weight * center_loss
        
        if self.base_model.losses:
            reg_loss = tf.add_n(self.base_model.losses)
        else:
            reg_loss = tf.constant(0.0, dtype=total_loss.dtype)
        
        total_loss = total_loss + reg_loss
        
        self.arcface_loss_tracker.update_state(arcface_loss)
        self.center_loss_tracker.update_state(center_loss)
        self.total_loss_tracker.update_state(total_loss)
        self.accuracy_tracker.update_state(labels, logits)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "arcface_loss": self.arcface_loss_tracker.result(),
            "center_loss": self.center_loss_tracker.result(),
            "accuracy": self.accuracy_tracker.result()
        }
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.arcface_loss_tracker,
            self.center_loss_tracker,
            self.unknown_loss_tracker,
            self.accuracy_tracker
        ]
trainer = FaceOnlyTrainer(
    model, 
    center_loss_layer, 
    unknown_rejection_layer,
    CFG.CENTER_LOSS_WEIGHT,
    CFG.UNKNOWN_LOSS_WEIGHT
)

In [10]:
# CURRICULUM MIXING 

def make_curriculum_dataset(known_ds, unknown_ratio):
    """ Uses heavy augmentation on known samples """
    if unknown_ratio == 0.0:
        return known_ds.batch(CFG.BATCH_SIZE).prefetch(CFG.AUTOTUNE)
    
    # Enforce minimum known count
    n_known = max(4, int(CFG.BATCH_SIZE * (1.0 - unknown_ratio)))
    n_unknown = CFG.BATCH_SIZE - n_known
    
    print(f"  Batch composition: {n_known} known + {n_unknown} unknown")
    
    known_batched = known_ds.batch(n_known + n_unknown)
    
    def create_mixed_batch(inputs, labels):  # Accept both inputs and labels
        # inputs is a dict with 'face_input' and 'label_input'
        face_imgs = inputs['face_input']
        
        # Split into known and source for unknowns
        known_faces = face_imgs[:n_known]
        known_labels = labels[:n_known]
        
        unknown_source = face_imgs[n_known:n_known + n_unknown]
        
        # Create unknowns via heavy augmentation
        unknown_faces = tf.map_fn(augment_unknown, unknown_source, dtype=tf.float32)
        unknown_labels = tf.fill([n_unknown], -1)
        
        # Merge
        merged_faces = tf.concat([known_faces, unknown_faces], axis=0)
        merged_labels = tf.concat([known_labels, unknown_labels], axis=0)
        
        # Shuffle
        idx = tf.random.shuffle(tf.range(tf.shape(merged_labels)[0]))
        merged_faces = tf.gather(merged_faces, idx)
        merged_labels = tf.gather(merged_labels, idx)
        
        merged_inputs = {
            'face_input': merged_faces,
            'label_input': merged_labels
        }
        
        return merged_inputs, merged_labels
    
    return known_batched.map(create_mixed_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [12]:
# SANITY CHECK CALLBACK

class EmbeddingSanityCheckCallback(keras.callbacks.Callback):
    def __init__(self, train_data_builder, check_every=5):
        super().__init__()
        self.train_builder = train_data_builder
        self.check_every = check_every
        
        cow_ids = list(self.train_builder.split_data.keys())
        valid_cows = [c for c in cow_ids if len(self.train_builder.split_data[c]) >= 2]
        
        if len(valid_cows) < 2:
            print("warning: Not enough cows for sanity check")
            self.can_check = False
            return
        
        self.can_check = True
        self.cow_0 = valid_cows[0]
        self.cow_1 = valid_cows[1]
        
        self.img_0a = self.train_builder.split_data[self.cow_0][0]
        self.img_0b = self.train_builder.split_data[self.cow_0][1]
        self.img_1a = self.train_builder.split_data[self.cow_1][0]
        self.img_1b = self.train_builder.split_data[self.cow_1][1]
        
        print(f"  Sanity check: {self.cow_0} vs {self.cow_1}")
    
    def on_epoch_end(self, epoch, logs=None):
        if not self.can_check or (epoch + 1) % self.check_every != 0:
            return
        
        print(f"\n  [Sanity Check @ Epoch {epoch+1}]")
        
        img_0a = load_image(self.img_0a).numpy()
        img_0b = load_image(self.img_0b).numpy()
        img_1a = load_image(self.img_1a).numpy()
        img_1b = load_image(self.img_1b).numpy()
        
        images = np.stack([img_0a, img_0b, img_1a, img_1b])
        
        test_batch = {
            'face_input': images,
            'label_input': np.array([0, 0, 1, 1], dtype=np.int32)
        }
        
        _, embeddings = self.model.base_model(test_batch, training=False)
        emb = embeddings.numpy()
        
        sim_0 = np.dot(emb[0], emb[1])
        sim_1 = np.dot(emb[2], emb[3])
        intra_sim = (sim_0 + sim_1) / 2
        
        sim_01 = np.dot(emb[0], emb[2])
        sim_02 = np.dot(emb[1], emb[3])
        inter_sim = (sim_01 + sim_02) / 2
        
        margin = intra_sim - inter_sim
        margin_status = "ok" if margin > 0.1 else "warning" if margin > 0 else "incorrect"
        
        print(f"    Intra: {intra_sim:.4f} | Inter: {inter_sim:.4f} | Margin: {margin:.4f} [{margin_status}]")

In [13]:
# PHASED TRAINING 

print("CURRICULUM TRAINING")

print(f"\nPhase 1 (Epochs 1-{CFG.PHASE_1_EPOCHS}): 100% known")
print(f"Phase 2 (Epochs {CFG.PHASE_1_EPOCHS+1}-{CFG.PHASE_1_EPOCHS+CFG.PHASE_2_EPOCHS}): 80% known, 20% unknown")
print(f"Phase 3 (Epochs {CFG.PHASE_1_EPOCHS+CFG.PHASE_2_EPOCHS+1}-{CFG.TOTAL_EPOCHS}): 60% known, 40% unknown")
print(f"\nTotal epochs: {CFG.TOTAL_EPOCHS}")

# Validation dataset
val_ds = val_data.get_tfdata_unbatched(
    shuffle=False,
    augment=False
).batch(CFG.BATCH_SIZE).prefetch(CFG.AUTOTUNE).repeat()  # Added .repeat() like goat model

val_steps = int(np.ceil(val_data.total_images / CFG.BATCH_SIZE))

# Build model variables by calling it with dummy data - This is required for subclassed models before loading weights

print("\n Building model variables...")
dummy_batch = next(iter(val_ds))
_ = trainer(dummy_batch[0], training=False)
print(" Model variables created")

# PHASE 1: WARMUP (Known only)

print(f"PHASE 1: Warmup (Epochs 1-{CFG.PHASE_1_EPOCHS})")

base_ds_p1 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p1 = make_curriculum_dataset(base_ds_p1, unknown_ratio=0.0)
trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))
phase1_callbacks = [
    keras.callbacks.ModelCheckpoint(
        'cattle_phase1_best.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        mode='min',
        verbose=1
    ),
    EmbeddingSanityCheckCallback(train_data, check_every=2)
]
history_p1 = trainer.fit(
    train_ds_p1,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    epochs=CFG.PHASE_1_EPOCHS,
    callbacks=phase1_callbacks,
    verbose=1
)
print(f"\nPhase 1 complete. Best val_loss: {min(history_p1.history['val_loss']):.4f}")

# PHASE 2: INTRODUCTION (80% known, 20% unknown)


print(f"PHASE 2: Unknown Introduction (Epochs {CFG.PHASE_1_EPOCHS+1}-{CFG.PHASE_1_EPOCHS+CFG.PHASE_2_EPOCHS})")

base_ds_p2 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p2 = make_curriculum_dataset(base_ds_p2, unknown_ratio=0.2)

# Load best weights from Phase 1
trainer.load_weights('cattle_phase1_best.weights.h5')

phase2_callbacks = [
    keras.callbacks.ModelCheckpoint(
        'cattle_phase2_best.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        mode='min',
        verbose=1
    ),
    EmbeddingSanityCheckCallback(train_data, check_every=3)
]
history_p2 = trainer.fit(
    train_ds_p2,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    initial_epoch=CFG.PHASE_1_EPOCHS,
    epochs=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    callbacks=phase2_callbacks,
    verbose=1
)
print(f"\nPhase 2 complete. Best val_loss: {min(history_p2.history['val_loss']):.4f}")

# PHASE 3: HARDENING (60% known, 40% unknown, Lower LR)

print(f"PHASE 3: Hardening (Epochs {CFG.PHASE_1_EPOCHS+CFG.PHASE_2_EPOCHS+1}-{CFG.TOTAL_EPOCHS})")


base_ds_p3 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p3 = make_curriculum_dataset(base_ds_p3, unknown_ratio=0.4)

# Load best weights from Phase 2
trainer.load_weights('cattle_phase2_best.weights.h5')
# Lower LR for Phase 3

trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4))
phase3_callbacks = [
    keras.callbacks.ModelCheckpoint(
        'cattle_final_best.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        mode='min',
        verbose=1
    ),
    EmbeddingSanityCheckCallback(train_data, check_every=3)
]
history_p3 = trainer.fit(
    train_ds_p3,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=val_steps,
    initial_epoch=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    epochs=CFG.TOTAL_EPOCHS,
    callbacks=phase3_callbacks,
    verbose=1
)

print(f"\nPhase 3 complete. Best val_loss: {min(history_p3.history['val_loss']):.4f}")

print("TRAINING COMPLETE")

CURRICULUM TRAINING

Phase 1 (Epochs 1-5): 100% known
Phase 2 (Epochs 6-20): 80% known, 20% unknown
Phase 3 (Epochs 21-30): 60% known, 40% unknown

Total epochs: 30

 Building model variables...
 Model variables created
PHASE 1: Warmup (Epochs 1-5)
  Sanity check: DEONI-19-C2082 vs GB-5-C8012
Epoch 1/5
200/200 [==============================] - ETA: 0s - loss: 25.3354 - arcface_loss: 25.1588 - center_loss: 0.8265 - unknown_loss: 0.0000e+00 - accuracy: 0.0269
Epoch 1: val_loss improved from inf to 11.02457, saving model to cattle_phase1_best.weights.h5
200/200 [==============================] - 28s 85ms/step - loss: 25.3354 - arcface_loss: 25.1588 - center_loss: 0.8265 - unknown_loss: 0.0000e+00 - accuracy: 0.0269 - val_loss: 11.0246 - val_arcface_loss: 10.8386 - val_center_loss: 0.6755 - val_accuracy: 0.2188
Epoch 2/5
200/200 [==============================] - ETA: 0s - loss: 6.4218 - arcface_loss: 6.2291 - center_loss: 0.6534 - unknown_loss: 0.0000e+00 - accuracy: 0.3692
Epoch 2: val_

In [14]:
# ANALYZE UNKNOWN REJECTION

print("ANALYZING UNKNOWN REJECTION")

# Load best final weights
trainer.load_weights('cattle_final_best.weights.h5')

# Get ArcFace weights (class centers)
arcface_layer = model.get_layer('arcface')
class_weights = arcface_layer.W.numpy()
class_weights_norm = class_weights / np.linalg.norm(class_weights, axis=0, keepdims=True)
print(f"\nClass centers shape: {class_weights_norm.shape}")

# Sample validation images and create synthetic unknowns
val_sample = next(iter(val_ds))
val_inputs, val_labels = val_sample

# Take first 20 images
n_test = min(20, CFG.BATCH_SIZE)
known_imgs = val_inputs['face_input'][:n_test].numpy()

# Create heavy augmented versions as synthetic unknowns
unknown_imgs = np.array([augment_unknown(tf.constant(img)).numpy() for img in known_imgs])

# Get embeddings
known_batch = {
    'face_input': tf.constant(known_imgs),
    'label_input': tf.constant([0]*n_test, dtype=tf.int32)
}
unknown_batch = {
    'face_input': tf.constant(unknown_imgs),
    'label_input': tf.constant([-1]*n_test, dtype=tf.int32)
}
_, known_emb = model(known_batch, training=False)
_, unknown_emb = model(unknown_batch, training=False)

# Compute max similarities
known_sim = np.matmul(known_emb.numpy(), class_weights_norm)
unknown_sim = np.matmul(unknown_emb.numpy(), class_weights_norm)
known_max = np.max(known_sim, axis=1)
unknown_max = np.max(unknown_sim, axis=1)
print(f"\nMax similarity to closest class:")
print(f"  Known samples:   {known_max.mean():.4f} ± {known_max.std():.4f}")
print(f"  Unknown samples: {unknown_max.mean():.4f} ± {unknown_max.std():.4f}")
print(f"  Separation:      {known_max.mean() - unknown_max.mean():.4f}")
separation = known_max.mean() - unknown_max.mean()
if separation > 0.15:
    print(" GOOD separation between known and unknown")
elif separation > 0.05:
    print(" WEAK separation - may need higher UNKNOWN_LOSS_WEIGHT")
else:
    print(" NO separation - unknown rejection likely won't work")

# Suggest threshold
suggested_threshold = (known_max.mean() + unknown_max.mean()) / 2
print(f"\nSuggested initial threshold: {suggested_threshold:.4f}")

ANALYZING UNKNOWN REJECTION

Class centers shape: (512, 468)

Max similarity to closest class:
  Known samples:   0.7313 ± 0.1109
  Unknown samples: 0.1501 ± 0.0693
  Separation:      0.5812
 GOOD separation between known and unknown

Suggested initial threshold: 0.4407


In [15]:
# MODEL EXPORT

print("EXPORTING MODEL FOR INFERENCE")

print("\nCreating inference model...")
face_inference = keras.Model(
    inputs=trainer.base_model.input[0],
    outputs=trainer.base_model.output[1],
    name="cattle_face_inference"
)

print("\nTesting inference model...")
test_img = tf.random.normal((2, 224, 224, 3))
test_emb = face_inference(test_img, training=False)
print(f"  Input shape: {test_img.shape}")
print(f"  Output shape: {test_emb.shape}")
print(f"  Norms: {tf.norm(test_emb, axis=1).numpy()}")
print("\nSaving as SavedModel...")

save_path = "cattle_face_curriculum_saved_model"
tf.saved_model.save(face_inference, save_path)
print(f" Saved to: {save_path}/")
print("\nTesting load...")

loaded = tf.saved_model.load(save_path)
infer = loaded.signatures["serving_default"]
test_output = infer(tf.constant(test_img))
output_key = list(test_output.keys())[0]
loaded_emb = test_output[output_key].numpy()
print(f"  Loaded shape: {loaded_emb.shape}")
print(f"  Loaded norms: {np.linalg.norm(loaded_emb, axis=1)}")

# Save class centers for threshold tuning
np.save('cattle_class_centers.npy', class_weights_norm.T)
print(f"\n  Saved class centers to: cattle_class_centers.npy")

# Save class mapping
import json
class_mapping = {
    'classes': train_data.classes,
    'num_classes': train_data.num_classes
}

with open('cattle_class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f" Saved class mapping to: cattle_class_mapping.json")


print("EXPORT COMPLETE")

#print("\n Saved artifacts:")
#print(f"  • Phase 1 weights: cattle_phase1_best.weights.h5")
#print(f"  • Phase 2 weights: cattle_phase2_best.weights.h5")
#print(f"  • Final weights:   cattle_final_best.weights.h5")
#print(f"  • Inference model: {save_path}/")
#print(f"  • Class centers:   cattle_class_centers.npy")
#print(f"  • Class mapping:   cattle_class_mapping.json")
#print("\n Architecture Summary:")
#print("  ✓ IDENTICAL to goat model")
#print("  ✓ EfficientNetB0 backbone with adaptive BatchNorm")
#print("  ✓ 512-dim L2-normalized embeddings")
#print("  ✓ ArcFace (margin=0.5, scale=64)")
#print("  ✓ Center Loss (weight=0.0001)")
#print("  ✓ Unknown Rejection Loss (weight=0.1)")
#print("\n Training Strategy:")
#print("  ✓ 3-phase curriculum learning")
#print("  ✓ Synthetic unknowns via heavy augmentation")
#print("  ✓ Fixed epoch boundaries (no early stopping)")
#print("  ✓ Progressive unknown exposure: 0% → 20% → 40%")

EXPORTING MODEL FOR INFERENCE

Creating inference model...

Testing inference model...
  Input shape: (2, 224, 224, 3)
  Output shape: (2, 512)
  Norms: [0.99999994 0.99999994]

Saving as SavedModel...


INFO:tensorflow:Assets written to: cattle_face_curriculum_saved_model/assets


INFO:tensorflow:Assets written to: cattle_face_curriculum_saved_model/assets


 Saved to: cattle_face_curriculum_saved_model/

Testing load...
  Loaded shape: (2, 512)
  Loaded norms: [0.9999999 1.       ]

  Saved class centers to: cattle_class_centers.npy
 Saved class mapping to: cattle_class_mapping.json
EXPORT COMPLETE
